In [ ]:
# Build the class dataset: the full Parquet (benign + attack) converted to one CSV.
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("build_class_dataset")
         .config("spark.driver.memory", "8g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

base = ("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/correlated_dataset/"
        "2026-09-26_00-28-45/finalized_data")
out  = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"

# Column order taken from the sample CSV header, so the file matches what the notebook expects
cols = spark.read.option("header", True).csv(base + "/Sample_Dataset_CSV/Benign/part-*.csv").columns

full = spark.read.parquet(base + "/Full_Dataset_Parquet/2026-*")
full = full.withColumn("datetime", F.date_format("datetime", "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
full = full.select([(F.col(c).cast("string") if c in full.columns
                     else F.lit(None).cast("string")).alias(c) for c in cols])

print("Total rows:", full.count())
full.groupBy("label_tactic").count().orderBy("count", ascending=False).show(truncate=False)

full.coalesce(1).write.mode("overwrite").option("header", True).csv(out)
print("Written to", out)

In [ ]:
# pull attack output to compare to mission logs
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

df = (spark.read.option("header", True)
      .csv("hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/class_dataset_full_csv"))
summary = (df.filter((F.col("label_tactic") != "none") & (F.col("label_technique") != "Duplicate"))
             .withColumn("day", F.substring("datetime", 1, 10))
             .groupBy("src_ip_zeek", "dest_ip_zeek", "label_technique", "label_tactic", "day")
             .agg(F.count(F.lit(1)).alias("connections"),
                  F.countDistinct("dest_port_zeek").alias("distinct_dest_ports"),
                  F.min("datetime").alias("first"), F.max("datetime").alias("last"))
             .orderBy("first"))
pdf = summary.toPandas()
pdf.to_csv("/home/ubuntu/CNT4416_attack_summary.csv", index=False)
print(len(pdf), "rows")
#

In [1]:
#test missing entries
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "6g").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "UTC")

logs = "hdfs://192.168.1.250:9000/tmp/82913-CNT4416-202608/nsm/zeek/logs"

checks = [
    (4,  "143.88.2.13",  "143.88.2.1",    "2026-09-01T20:56:01Z", "2026-09-01T20:56:06Z"),  # G2 T1595 443
    (6,  "143.88.2.13",  "143.88.2.11",   "2026-09-01T20:57:30Z", "2026-09-01T20:57:35Z"),  # G2 T1595 80
    (7,  "143.88.2.13",  "143.88.2.12",   "2026-09-01T20:57:38Z", "2026-09-01T20:57:40Z"),  # G2 T1595 80
    (8,  "143.88.2.13",  "143.88.2.13",   "2026-09-01T20:57:48Z", "2026-09-01T20:57:50Z"),  # G2 T1595 8000 (dst = src)
    (16, "143.88.4.13",  "143.88.255.11", "2026-08-30T15:22:01Z", "2026-08-30T15:22:09Z"),  # G4 T1595 80
    (28, "143.88.6.13",  "143.88.6.12",   "2026-09-08T15:15:36Z", "2026-09-08T15:15:42Z"),  # G5 T1046 (src is a group 6 IP)
    (35, "143.88.8.10",  "143.88.9.10",   "2026-09-09T19:41:05Z", "2026-09-09T19:41:20Z"),  # G8 T1595
    (36, "143.88.6.10",  "143.88.7.",     "2026-09-14T14:59:00Z", "2026-09-14T15:01:00Z"),  # G6 T1595
    (47, "143.88.9.13",  "143.88.10.12",  "2026-09-17T00:10:22Z", "2026-09-17T00:10:22Z"),  # G9 T1210 445
    (54, "143.88.1.11",  "143.88.10.10",  "2026-09-22T13:50:00Z", "2026-09-22T15:05:00Z"),  # G10 T1564 4446
    (55, "143.88.10.10", "143.88.1.18",   "2026-09-22T13:50:00Z", "2026-09-22T14:29:00Z"),  # G10 T1070 4444
    (56, "143.88.4.13",  "143.88.5.12",   "2026-09-22T20:56:41Z", "2026-09-22T20:56:57Z"),  # G4 T1134 445
    (57, "143.88.9.10",  "143.88.10.12",  "2026-09-22T17:37:53Z", "2026-09-22T18:15:00Z"),  # G9 T1210 445
    (58, "143.88.9.10",  "143.88.10.11",  "2026-09-22T18:18:26Z", "2026-09-22T18:25:00Z"),  # G9 T1070 21
    (60, "143.88.4.10",  "143.88.4.12",   "2026-09-23T13:59:47Z", "2026-09-23T14:00:17Z"),  # G3 T1210 445 (pivot)
    (63, "143.88.2.10",  "143.88.3.12",   "2026-09-23T21:23:00Z", "2026-09-23T21:41:00Z"),  # G2 T1070 445
    (66, "143.88.1.17",  "143.88.2.12",   "2026-09-24T03:44:47Z", "2026-09-24T04:35:34Z"),  # G1 T1070
    (67, "143.88.7.13",  "143.88.8.11",   "2026-09-23T23:39:00Z", "2026-09-23T23:40:00Z"),  # G7 T1070 21
]
def epoch(s): return datetime.fromisoformat(s.replace("Z", "+00:00")).timestamp()

# Only the day folders that exist for these entries (the day before through the day after)
days = sorted({(datetime.fromisoformat(s[:10]) + timedelta(days=k)).strftime("%Y-%m-%d")
               for _, _, _, s, _ in checks for k in (-1, 0, 1)})
jvm, conf = spark._jvm, spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.Path(logs).getFileSystem(conf)
paths = [f"{logs}/{d}/conn.*.log.gz" for d in days
         if fs.globStatus(jvm.org.apache.hadoop.fs.Path(f"{logs}/{d}/conn.*.log.gz"))]
print("Reading", len(paths), "day folders")

conn = (spark.read.json(paths)
        .select(F.col("`id.orig_h`").alias("src"), F.col("`id.resp_h`").alias("dst"),
                F.col("`id.resp_p`").alias("dport"),
                F.coalesce(F.col("ts").cast("double"), F.to_timestamp("ts").cast("double")).alias("ts"))
        .cache())

windows = [("as logged (±2 min)", 0, 120), ("+5 h (local as UTC)", 5 * 3600, 120),
           ("whole day (±12 h)", 0, 12 * 3600)]
for line, src, dst, start, end in checks:
    dmatch = F.col("dst").startswith(dst) if dst.endswith(".") else (F.col("dst") == dst)
    pair = conn.filter((F.col("src") == src) & dmatch)
    print(f"\nLine {line}: {src} -> {dst}  logged {start}")
    for name, shift, pad in windows:
        r = (pair.filter(F.col("ts").between(epoch(start) + shift - pad, epoch(end) + shift + pad))
                 .agg(F.count(F.lit(1)).alias("n"),
                      F.array_sort(F.collect_set("dport")).alias("ports"),
                      F.from_unixtime(F.min("ts").cast("long")).alias("first"))
                 .first())
        ports = ", ".join(str(p) for p in (r["ports"] or [])[:8])
        print(f"  {name:22s} {r['n']:>6} conns  ports={ports}  first={r['first']}")
    any_src = conn.filter((F.col("src") == src)
                          & F.col("ts").between(epoch(start) - 43200, epoch(end) + 43200)).count()
    print(f"  {'any traffic from src':22s} {any_src:>6} conns that day (to anywhere)")

26/10/26 17:22:19 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.131 instead (on interface ens33)
26/10/26 17:22:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 17:22:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/26 17:22:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/26 17:22:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Reading 12 day folders



Line 53: 143.88.5.10 -> 143.88.6.11  logged 2026-09-17T03:20:53Z


  as logged (±2 min)         10 conns  ports=21, 80, 59087  first=2026-09-17 03:20:53
  +5 h (local as UTC)         0 conns  ports=  first=None
  whole day (±12 h)      113393 conns  ports=4, 5, 6, 7, 8, 9, 10, 11  first=2026-09-17 02:55:13
  any traffic from src   170074 conns that day (to anywhere)

Line 20: 143.88.3.10 -> 143.88.3.12  logged 2026-09-05T12:36:08Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)      1189 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-05 17:35:47
  whole day (±12 h)        2261 conns  ports=1, 3, 4, 6, 7, 9, 13, 17  first=2026-09-05 17:12:40
  any traffic from src    68591 conns that day (to anywhere)

Line 21: 143.88.1.10 -> 143.88.1.11  logged 2026-09-08T14:40:00Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)         0 conns  ports=  first=None
  whole day (±12 h)           0 conns  ports=  first=None
  any traffic from src        0 conns that day (to anywhere)

Line 22: 143.88.4

  whole day (±12 h)        4968 conns  ports=0, 1, 3, 4, 6, 7, 9, 13  first=2026-09-09 21:14:58
  any traffic from src    61245 conns that day (to anywhere)

Line 50: 143.88.4.10 -> 143.88.5.11  logged 2026-09-17T01:35:53Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)         0 conns  ports=  first=None
  whole day (±12 h)          24 conns  ports=21, 80  first=2026-09-16 23:52:34
  any traffic from src    55779 conns that day (to anywhere)

Line 51: 143.88.7.10 -> 143.88.8.11  logged 2026-09-17T22:03:47Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)         0 conns  ports=  first=None
  whole day (±12 h)           0 conns  ports=  first=None
  any traffic from src        0 conns that day (to anywhere)

Line 52: 143.88.7.10 -> 143.88.8.12  logged 2026-09-17T22:15:53Z
  as logged (±2 min)          0 conns  ports=  first=None
  +5 h (local as UTC)         0 conns  ports=  first=None
  whole day (±12 h)           0 conns 